<a href="https://colab.research.google.com/github/devpalavicini/sistema_integrado_inteligente_colonia/blob/main/Fase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dados historicos usados para prever a geracao eolica.
ventos_historicos = [8, 10, 12]
geracao_historica = [20, 25, 30]


def criar_colonia():  # Dados da colonia organizados em um dicionario.
    colonia = {
        "energia": {
            "solar": 20,
            "eolica": 0,
            "reserva": 100,
            "capacidade_bateria": 200
        },
        "clima": {
            "vento": 11,
            "tempestade_areia": False
        },
        "modulos": {
            "Habitacao e suporte de vida": {
                "consumo": 20,
                "ativo": True,
                "prioridade": "essencial"
            },
            "Suporte medico": {
                "consumo": 15,
                "ativo": True,
                "prioridade": "essencial"
            },
            "Laboratorio": {
                "consumo": 15,
                "ativo": True,
                "prioridade": "nao essencial"
            },
            "Logistica": {
                "consumo": 10,
                "ativo": True,
                "prioridade": "nao essencial"
            },
            "Mineracao": {
                "consumo": 20,
                "ativo": True,
                "prioridade": "nao essencial"
            }
        }
    }
    return colonia


def prever_energia_eolica(vento, ventos, geracoes): # Calcula uma reta simples de regressao usando os dados historicos.
    quantidade = len(ventos)
    soma_vento = 0
    soma_geracao = 0
    soma_multiplicacao = 0
    soma_vento_quadrado = 0

    for i in range(quantidade):
        x = ventos[i]
        y = geracoes[i]
        soma_vento = soma_vento + x
        soma_geracao = soma_geracao + y
        soma_multiplicacao = soma_multiplicacao + (x * y)
        soma_vento_quadrado = soma_vento_quadrado + (x * x)

    inclinacao = (quantidade * soma_multiplicacao - soma_vento * soma_geracao) / (quantidade * soma_vento_quadrado - soma_vento * soma_vento)
    ponto_inicial = (soma_geracao - inclinacao * soma_vento) / quantidade
    previsao = ponto_inicial + inclinacao * vento

    if previsao < 0:
        previsao = 0

    return previsao


def calcular_consumo(modulos): #Função para calcular o consumo dos modulos
    consumo_total = 0

    for nome in modulos:
        if modulos[nome]["ativo"]:
            consumo_total = consumo_total + modulos[nome]["consumo"]

    return consumo_total


def ativar_modo_economia(modulos): #Função para ativar o modo economia
    print("MODO ECONOMIA: desligando os modulos nao essenciais.")

    for nome in modulos:
        if modulos[nome]["prioridade"] == "nao essencial":
            modulos[nome]["ativo"] = False
            print("Desligado:", nome)
        else:
            print("Mantido ligado:", nome)


def analisar_reserva(colonia, geracao_total, consumo_total): #Função para analisar a reserva energética
    reserva = colonia["energia"]["reserva"]
    capacidade = colonia["energia"]["capacidade_bateria"]

    if consumo_total > geracao_total:
        deficit = consumo_total - geracao_total
        print("ALERTA: consumo maior que a geracao.")

        if reserva >= deficit:
            reserva = reserva - deficit
            print("Acao: usando a bateria para cobrir o deficit de", round(deficit, 2))
        else:
            print("ALERTA CRITICO: a reserva nao e suficiente.")
            reserva = 0
    else:
        if geracao_total > consumo_total:
            excedente = geracao_total - consumo_total
            print("SUGESTAO: armazenar energia excedente de", round(excedente, 2))

            if reserva + excedente > capacidade:
                reserva = capacidade
                print("Bateria cheia. Parte do excedente nao pode ser guardada.")
            else:
                reserva = reserva + excedente
                print("Excedente armazenado na bateria.")
        else:
            print("BALANCO: geracao igual ao consumo.")

    colonia["energia"]["reserva"] = reserva
    print("Reserva final:", round(reserva, 2))


def mostrar_modulos(modulos):
    print("\nSTATUS DOS MODULOS")

    for nome in modulos:
        if modulos[nome]["ativo"]:
            print(nome, "- ATIVO - consumo:", modulos[nome]["consumo"])
        else:
            print(nome, "- DESLIGADO - consumo:", modulos[nome]["consumo"])


def executar_sistema(colonia):
    vento = colonia["clima"]["vento"]
    previsao_eolica = prever_energia_eolica(vento, ventos_historicos, geracao_historica)
    colonia["energia"]["eolica"] = previsao_eolica

    geracao_solar = colonia["energia"]["solar"]
    if colonia["clima"]["tempestade_areia"]:
        geracao_solar = geracao_solar / 2
        print("ALERTA: tempestade de areia. Geracao solar reduzida pela metade.")

    geracao_total = geracao_solar + colonia["energia"]["eolica"]
    consumo_inicial = calcular_consumo(colonia["modulos"])

    print("\n=== SISTEMA DA COLONIA AURORA SIGER ===")
    print("Velocidade do vento:", vento)
    print("Previsao de geracao eolica:", round(previsao_eolica, 2))
    print("Geracao solar:", round(geracao_solar, 2))
    print("Geracao total:", round(geracao_total, 2))
    print("Consumo inicial:", round(consumo_inicial, 2))

    if geracao_total < 30:
        print("ALERTA CRITICO: geracao abaixo de 30.")
        ativar_modo_economia(colonia["modulos"])
    else:
        if geracao_total < 50:
            print("ALERTA: geracao abaixo de 50. Reduzindo consumo.")
            ativar_modo_economia(colonia["modulos"])
        else:
            if consumo_inicial > geracao_total:
                print("ALERTA: consumo maior que a geracao. Reduzindo consumo.")
                ativar_modo_economia(colonia["modulos"])
            else:
                print("Geracao suficiente para o consumo inicial.")

    consumo_final = calcular_consumo(colonia["modulos"])
    print("Consumo depois das decisoes:", round(consumo_final, 2))
    analisar_reserva(colonia, geracao_total, consumo_final)
    mostrar_modulos(colonia["modulos"])


# Programa principal
continuar = "s"
while continuar == "s":
    colonia = criar_colonia()

    vento = float(input("Digite a velocidade do vento (exemplo: 11): "))
    while vento < 0:
        print("A velocidade do vento nao pode ser negativa.")
        vento = float(input("Digite a velocidade do vento novamente: "))

    solar = float(input("Digite a geracao solar (exemplo: 20): "))
    while solar < 0:
        print("A geracao solar nao pode ser negativa.")
        solar = float(input("Digite a geracao solar novamente: "))

    tempestade = input("Ha tempestade de areia? (s/n): ")
    if tempestade == "s":
        colonia["clima"]["tempestade_areia"] = True
    else:
        colonia["clima"]["tempestade_areia"] = False

    colonia["clima"]["vento"] = vento
    colonia["energia"]["solar"] = solar
    executar_sistema(colonia)

    continuar = input("\nDeseja fazer outra simulacao? (s/n): ")